In [1]:
%pip install pandas nltk gensim pyLDAvis

import pandas as pd

import sklearn
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models import CoherenceModel
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package wordnet to /Users/Licas/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/Licas/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# chargement du dataset allocine

In [3]:
%pip install datasets 
from datasets import load_dataset 
import pandas as pd 
dataset = load_dataset('tblard/allocine', split='train') 
df = dataset.to_pandas() # Exemple d’échantillon pour travailler plus vite df = df.sample(n=3000, random_state=42).reset_index(drop=True) print(df.shape) df[['review', 'label']].head()

  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached pyarrow-24.0.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.19-py313-none-any.whl.metadata (7.5 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached multidict-6.7.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.3 kB)
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached multiprocess-0.70.19-py313-none-any.whl (156 kB)
Using cached pyarrow-24.0.0-cp313-cp313-macosx_12_0_arm64.whl (35.0 MB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached frozenlist-1.8.0-cp313-cp313-macosx_11_0_arm64.whl (49 kB)
Using cached multidict-6.7.1-cp313-cp313-macosx_11_0_arm64.whl (43 kB)

[notice] A new release of pip is available: 25.0.1 -

/Users/Licas/Desktop/github/NLP/nlpenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 20000/20000 [00:00<00:00, 1158279.55 examples/s]


In [4]:
df = df.sample(n=3000, random_state=42).reset_index(drop=True) 
print(df.shape) 
df[['review', 'label']].head()

(3000, 2)


,review,label
0,Un excellent thriller d'action où les scènes d...,1
1,"Si le scénariste, qui aurait pu faire un minim...",0
2,"Référence dans la filmographie de Bogart, ""Le ...",0
3,"Un bon scénario, un bon film, une histoire lou...",1
4,Un scenario vide et une mise en scene trés sop...,0


In [5]:
liste_textes = df["review"].dropna().tolist()
print(liste_textes[0])

Un excellent thriller d'action où les scènes de bravoure se croisent avec des effets pyrotechniques sensationnelle. Une course contre la montre haletante au rythme effréné, impossible de décrocher !


# Pre processing 

In [6]:
stemmer = SnowballStemmer('french')

stopwords_fr= STOPWORDS.union({
'les','des','une','dans','pour','avec','après','avant',
'plus','tout','tous','être','avoir'
})

def lemmatize_stemming(token):
    lemme= WordNetLemmatizer().lemmatize(token, pos='n')
    return stemmer.stem(lemme)

def preprocess(text):
    tokens = []
    for token in simple_preprocess(text, deacc=True):
        if token not in stopwords_fr and len(token) > 3:
            tokens.append(lemmatize_stemming(token))
    return tokens

In [7]:
processed_docs= [preprocess(doc) for doc in liste_textes]

for titre, tokens in zip(df['label'], processed_docs):
    print(titre, '->', tokens)

1 -> ['excellent', 'thrill', 'action', 'scen', 'bravour', 'croisent', 'effet', 'pyrotechn', 'sensationnel', 'cours', 'contr', 'montr', 'halet', 'rythm', 'effren', 'impossibl', 'decroch']
0 -> ['scenar', 'aur', 'fair', 'minimum', 'recherch', 'evit', 'film', 'soit', 'rempl', 'erreur', 'histor', 'voul', 'provoqu', 'frisson', 'rat', 'certain', 'scen', 'font', 'mem', 'sourir', 'tel', 'elle', 'sont', 'rate', 'bon', 'ide', 'elle', 'sont', 'exploite', 'outr', 'actric', 'principal', 'credibl', 'surjou', 'possibl', 'revanch', 'film', 'deconseil', 'person', 'souffr', 'claustrophob', 'peut', 'etre', 'seul', 'interet', 'film', 'don', 'envi', 'visit', 'catacomb', 'paris']
0 -> ['referent', 'filmograph', 'bogart', 'violent', 'rest', 'moin', 'decev', 'usag', 'personnag', 'interess', 'evolu', 'scenario', 'plat', 'san', 'grand', 'rebond', 'nous', 'offrant', 'histoir', 'premier', 'semblabl', 'pet', 'hitchcock', 'nichol', 'laiss', 'entrain', 'banal', 'histoir', 'cœur', 'dont', 'seul', 'originalit', 'carac

# dico Gensim

In [13]:
dictionary= gensim.corpora.Dictionary(processed_docs)

print(dictionary.token2id)
print('Nombre de mots gardés :', len(dictionary))

{'action': 0, 'bravour': 1, 'contr': 2, 'cours': 3, 'croisent': 4, 'decroch': 5, 'effet': 6, 'effren': 7, 'excellent': 8, 'halet': 9, 'impossibl': 10, 'montr': 11, 'pyrotechn': 12, 'rythm': 13, 'scen': 14, 'sensationnel': 15, 'thrill': 16, 'actric': 17, 'aur': 18, 'bon': 19, 'catacomb': 20, 'certain': 21, 'claustrophob': 22, 'credibl': 23, 'deconseil': 24, 'don': 25, 'elle': 26, 'envi': 27, 'erreur': 28, 'etre': 29, 'evit': 30, 'exploite': 31, 'fair': 32, 'film': 33, 'font': 34, 'frisson': 35, 'histor': 36, 'ide': 37, 'interet': 38, 'mem': 39, 'minimum': 40, 'outr': 41, 'paris': 42, 'person': 43, 'peut': 44, 'possibl': 45, 'principal': 46, 'provoqu': 47, 'rat': 48, 'rate': 49, 'recherch': 50, 'rempl': 51, 'revanch': 52, 'scenar': 53, 'seul': 54, 'soit': 55, 'sont': 56, 'souffr': 57, 'sourir': 58, 'surjou': 59, 'tel': 60, 'visit': 61, 'voul': 62, 'acteur': 63, 'ains': 64, 'aujourd': 65, 'banal': 66, 'bien': 67, 'bobin': 68, 'bogart': 69, 'caracter': 70, 'cinem': 71, 'classiqu': 72, 'cor

In [14]:
# filtre avec valeurs extrèmes


dictionary.filter_extremes(
no_below=10, # garder les mots présents dans au moins 1 document
no_above=0.4, # supprimer les mots présents dans plus de 80% des documents
keep_n=5000 # garder au maximum 1000 mots
)
print('Vocabulaire final :', len(dictionary))

Vocabulaire final : 2029


# bag of words

In [15]:
bow_corpus= [dictionary.doc2bow(doc) for doc in processed_docs]
for i, bow in enumerate(bow_corpus[:3]):
    print(df.loc[i, 'label'], '->', bow)

1 -> [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)]
0 -> [(10, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 2), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1), (35, 1), (36, 1), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 2), (49, 1), (50, 1), (51, 1), (52, 1), (53, 1)]
0 -> [(2, 1), (10, 1), (14, 1), (33, 1), (46, 1), (54, 1), (55, 1), (56, 1), (57, 1), (58, 1), (59, 1), (60, 1), (61, 1), (62, 1), (63, 1), (64, 1), (65, 1), (66, 1), (67, 1), (68, 1), (69, 1), (70, 1), (71, 1), (72, 1), (73, 1), (74, 1), (75, 1), (76, 1), (77, 1), (78, 1), (79, 2), (80, 1), (81, 1), (82, 1), (83, 1), (84, 1), (85, 1), (86, 1), (87, 1), (88, 1), (89, 1), (90, 1), (91, 1), (92, 1), (93, 1), (94, 1), (95, 1), (96, 1), (97, 2), (98, 1), (99, 1), (100, 1), (101, 1), (102, 1), (103, 1),

In [26]:
# on crée une fonction réutilisable pour le model LDA, en faisant varier le nombre de topics directement en param
def training_LDA(n_max:int):
    for i in range(2, n_max):
        lda_model= gensim.models.LdaModel(
        corpus=bow_corpus,
        id2word=dictionary,
        num_topics=i,
        random_state=42,
        passes= 15,
        iterations=100,
        alpha='auto',
        eta='auto'
        )
        lda_model.save(f"modele_lda.gensim_{i}")
        dictionary.save("dictionnaire_lda.gensim")
        
        for idx, topic in lda_model.print_topics(num_words=10):
            print(f'Topic{idx} -> {topic}')

In [28]:
training_LDA(2)